# PPDONet-Stan: Densidad Superficial

Entrenamiento y evaluacion del modelo PPDONet para el mapa de densidad superficial `sigma`.

Los parametros fisicos (`ALPHA`, `PLANETMASS`, `ASPECTRATIO`) alimentan la red branch. Las coordenadas del mapa (`r`, `theta`) alimentan la red trunk mediante la transformacion periodica `(r_scaled, sin(theta), cos(theta))`.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

from ppdonet_common import *

device = get_device()
print_device_info(device)


## Datos

In [ ]:
train_dir, test_dir = download_planet2disk_data()
print("Train dir:", train_dir)
print("Test dir :", test_dir)

ds_sigma_tr, ds_vr_tr, ds_vtheta_tr = open_outputs(train_dir)
ds_sigma_te, ds_vr_te, ds_vtheta_te = open_outputs(test_dir)

params_tr, where_tr = find_and_load_params(train_dir, device)
params_te, where_te = find_and_load_params(test_dir, device)
print(where_tr)
print("params_tr:", params_tr.shape, params_tr.device)


In [ ]:
coords_tf, r_vals, th_vals = build_coords(ds_sigma_tr, device)
print("coords_tf:", coords_tf.shape, coords_tf.device)


## Dataset y loaders

In [ ]:
sigma_var = list(ds_sigma_tr.data_vars)[0]

NPOINTS_SAMPLE = 435483
BATCH_SIZE = 1

train_sigma_loader, test_sigma_loader = make_field_loaders(
    ds_sigma_tr,
    ds_sigma_te,
    params_tr,
    params_te,
    coords_tf,
    sigma_var,
    device,
    log_target=True,
    subtract_background=False,
    n_points_sample=NPOINTS_SAMPLE,
    batch_size=BATCH_SIZE,
)

u0, y0, t0 = next(iter(train_sigma_loader))
print("sample shapes (u, y, target):", u0.shape, y0.shape, t0.shape)
print("devices:", u0.device, y0.device, t0.device)


## Entrenamiento

In [ ]:
EPOCHS = 1500
LR = 1e-4
CHECKPOINT = "modelo_sigma_Stan.pt"

model_sigma = build_ppdonet(act="stan", stan_beta=0.1, stan_positive_beta=True)
hist_sigma, model_sigma = train_model(
    model_sigma,
    train_sigma_loader,
    test_sigma_loader,
    device,
    epochs=EPOCHS,
    lr=LR,
)
save_model(model_sigma, CHECKPOINT)


## Evaluacion

In [ ]:
CHECKPOINT = "modelo_sigma_Stan.pt"
model_sigma = load_model(CHECKPOINT, device)

mse_sigma, r2_sigma = evaluate_model(model_sigma, test_sigma_loader, device)
print(f"Sigma: MSE={mse_sigma:.4e}, R2={r2_sigma:.4f}")


## Ejemplo de test

In [ ]:
idx_example = 0

pred_sigma_map, true_sigma_map, mse_sigma_ex, r2_sigma_ex = predict_single_field(
    model_sigma,
    params_te,
    coords_tf,
    ds_sigma_te,
    sigma_var,
    r_vals,
    th_vals,
    device,
    idx_example=idx_example,
    log_target=True,
    subtract_background=False,
    fargo_kepler_delta=False,
)

print(f"Sigma ejemplo {idx_example}: MSE={mse_sigma_ex:.4e}, R2={r2_sigma_ex:.4f}")


In [ ]:
plot_field_maps(
    pred_sigma_map,
    true_sigma_map,
    r_vals,
    th_vals,
    title=f"Sigma (log10) - Test idx={idx_example}",
    cmap="viridis",
    robust=False,
)
